In [ ]:
# Cell 1: Install (fresh runtime recommended)
!pip install -q faster-whisper

# Cell 2: Load Kannada-tuned model
from faster_whisper import WhisperModel
import time

print("Loading Kannada model...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Model loaded")

# Cell 3: Upload and Transcribe
from google.colab import files

print("📁 Upload your Kannada audio file:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

start = time.time()
segments, info = model.transcribe(
    audio_path,
    language="kn",
    beam_size=5,
    best_of=5
)
segments_list = list(segments)
latency = (time.time() - start) * 1000

print(f"\n🇮🇳 ಕನ್ನಡ Transcription:")
full_text = ""
for seg in segments_list:
    print(f"   {seg.text}")
    full_text += seg.text

print(f"\n📝 Full: {full_text}")
print(f"⏱️ Latency: {latency:.0f}ms")

In [ ]:
# Cell 1: Install
!pip install -q faster-whisper

# Cell 2: Load Fast Model
from faster_whisper import WhisperModel
import time

print("Loading fast Kannada model...")
model = WhisperModel("medium", device="cuda", compute_type="int8_float16")
print("✓ Model loaded")

# Cell 3: Upload and Transcribe
from google.colab import files

print("📁 Upload your Kannada audio file:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

start = time.time()
segments, info = model.transcribe(audio_path, language="kn", beam_size=1)
segments_list = list(segments)
latency = (time.time() - start) * 1000

print(f"\n🇮🇳 ಕನ್ನಡ Transcription:")
full_text = ""
for seg in segments_list:
    print(f"   {seg.text}")
    full_text += seg.text

print(f"\n📝 Full: {full_text}")
print(f"⏱️ Latency: {latency:.0f}ms")

In [ ]:
# Cell 1: Install
!pip install -q faster-whisper

# Cell 2: Load Model
from faster_whisper import WhisperModel
import time

print("Loading Hindi model...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Model loaded")

# Cell 3: Upload and Transcribe Hindi
from google.colab import files

print("📁 Upload your Hindi audio file:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

start = time.time()
segments, info = model.transcribe(audio_path, language="hi", beam_size=5, best_of=5)
segments_list = list(segments)
latency = (time.time() - start) * 1000

print(f"\n🇮🇳 हिंदी Transcription:")
full_text = ""
for seg in segments_list:
    print(f"   {seg.text}")
    full_text += seg.text

print(f"\n📝 Full: {full_text}")
print(f"⏱️ Latency: {latency:.0f}ms")

In [ ]:
# Cell 1: Install
!pip install -q faster-whisper

# Cell 2: Load Model
from faster_whisper import WhisperModel
import time

print("Loading Telugu model...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Model loaded")

# Cell 3: Upload and Transcribe Telugu
from google.colab import files

print("📁 Upload your Telugu audio file:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

start = time.time()
segments, info = model.transcribe(audio_path, language="te", beam_size=5, best_of=5)
segments_list = list(segments)
latency = (time.time() - start) * 1000

print(f"\n🇮🇳 తెలుగు Transcription:")
full_text = ""
for seg in segments_list:
    print(f"   {seg.text}")
    full_text += seg.text

print(f"\n📝 Full: {full_text}")
print(f"⏱️ Latency: {latency:.0f}ms")

In [ ]:
# Cell: Fix torchaudio audio loading
!pip install -q torchcodec soundfile librosa

print("✓ Audio dependencies installed")

In [ ]:
# Cell: Load Silero VAD Model
import torch
torch.set_num_threads(1)  # Important for speed

SAMPLING_RATE = 16000

model_vad, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    onnx=False
)

(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils

print("✓ Silero VAD loaded")

In [ ]:
# Cell: Test VAD on Uploaded Audio (Fixed)
from google.colab import files
import time
import librosa
import numpy as np

print("📁 Upload an audio file to test VAD:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Read audio with librosa (works for MP3, WAV, etc.)
wav_np, sr = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True)
wav = torch.from_numpy(wav_np).float()

print(f"✓ Audio loaded: {len(wav)/SAMPLING_RATE:.2f} seconds")

# Run VAD
start = time.time()
speech_timestamps = get_speech_timestamps(
    wav,
    model_vad,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=600,
    sampling_rate=SAMPLING_RATE
)
vad_latency = (time.time() - start) * 1000

# Results
print(f"\n🎯 VAD Results:")
print(f"⏱️ VAD Latency: {vad_latency:.1f}ms")
print(f"🗣️ Speech segments found: {len(speech_timestamps)}")

for i, ts in enumerate(speech_timestamps):
    start_sec = ts['start'] / SAMPLING_RATE
    end_sec = ts['end'] / SAMPLING_RATE
    duration = end_sec - start_sec
    print(f"   Segment {i+1}: {start_sec:.2f}s → {end_sec:.2f}s ({duration:.2f}s)")

In [ ]:
# Cell: Complete VAD → ASR Pipeline
!pip install -q faster-whisper librosa

from faster_whisper import WhisperModel
import librosa
import torch
import time
import numpy as np

# Load ASR model
print("Loading ASR model...")
model_asr = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ ASR model loaded")

# Upload audio
from google.colab import files
print("\n📁 Upload audio for VAD + ASR pipeline:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Read audio with librosa
SAMPLING_RATE = 16000
wav_np, sr = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True)
wav = torch.from_numpy(wav_np).float()

# ===== STEP 1: VAD =====
start_vad = time.time()
speech_timestamps = get_speech_timestamps(
    wav, model_vad,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=600,
    sampling_rate=SAMPLING_RATE
)
vad_latency = (time.time() - start_vad) * 1000

print(f"\n🎯 VAD: {len(speech_timestamps)} segments detected ({vad_latency:.1f}ms)")

# ===== STEP 2: Extract speech & run ASR =====
import soundfile as sf

for i, ts in enumerate(speech_timestamps):
    # Extract speech segment
    speech_chunk = wav_np[ts['start']:ts['end']]

    # Save temp file for Whisper
    temp_path = f"segment_{i}.wav"
    sf.write(temp_path, speech_chunk, SAMPLING_RATE)

    # Run ASR
    start_asr = time.time()
    segments, info = model_asr.transcribe(
        temp_path,
        language="kn",  # Change to "hi" for Hindi
        beam_size=5
    )
    asr_latency = (time.time() - start_asr) * 1000

    # Get transcription
    text = " ".join([seg.text for seg in segments])

    start_sec = ts['start'] / SAMPLING_RATE
    end_sec = ts['end'] / SAMPLING_RATE

    print(f"\n📌 Segment {i+1} [{start_sec:.2f}s - {end_sec:.2f}s]")
    print(f"   📝 Transcription: {text.strip()}")
    print(f"   ⏱️ ASR Latency: {asr_latency:.0f}ms")

print("\n✅ VAD + ASR Pipeline Complete!")

In [ ]:
# Cell: Simulate Real-time VAD (Chunk Processing)
import time

# This simulates how VAD would work in a real-time streaming scenario

print("📁 Upload audio for real-time VAD simulation:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

wav = read_audio(audio_path, sampling_rate=SAMPLING_RATE)

# Chunk size: 30ms (512 samples at 16kHz)
CHUNK_SIZE = 512  # ~32ms chunks

# Initialize VAD iterator (streaming mode)
vad_iterator = VADIterator(model_vad, sampling_rate=SAMPLING_RATE)

# Simulate streaming
print(f"\n🔴 Simulating real-time VAD on {len(wav)/SAMPLING_RATE:.2f}s audio...")
print(f"   Chunk size: {CHUNK_SIZE/SAMPLING_RATE*1000:.0f}ms")

speech_buffer = []
is_speaking = False
speech_count = 0

for i in range(0, len(wav), CHUNK_SIZE):
    chunk = wav[i:i+CHUNK_SIZE]

    if len(chunk) < CHUNK_SIZE:
        # Pad last chunk if needed
        chunk = torch.nn.functional.pad(chunk, (0, CHUNK_SIZE - len(chunk)))

    # Run VAD on this chunk
    start = time.time()
    speech_dict = vad_iterator(chunk, return_seconds=True)
    latency = (time.time() - start) * 1000

    if speech_dict:
        if 'start' in speech_dict:
            print(f"   🟢 Speech START at {speech_dict['start']:.2f}s (latency: {latency:.1f}ms)")
            is_speaking = True
            speech_count += 1

        if 'end' in speech_dict:
            print(f"   🔴 Speech END at {speech_dict['end']:.2f}s (latency: {latency:.1f}ms)")
            is_speaking = False

# Reset for next use
vad_iterator.reset_states()

print(f"\n✅ Real-time simulation complete!")
print(f"   Total speech segments: {speech_count}")

In [ ]:
# ============================================================
# 🔧 FIXED: Force Language + Fallback Detection
# ============================================================

# Install dependencies
!pip install -q faster-whisper librosa soundfile torch

import time
import torch
import librosa
import soundfile as sf
from google.colab import files
from faster_whisper import WhisperModel

# Load models
print("Loading models...")
SAMPLING_RATE = 16000
torch.set_num_threads(1)

model_vad, utils = torch.hub.load('snakers4/silero-vad', model='silero_vad', force_reload=False, onnx=False)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

model_asr = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Models loaded")

# Supported languages
SUPPORTED_LANGS = {
    'kn': 'Kannada',
    'hi': 'Hindi',
    'ta': 'Tamil',
    'te': 'Telugu',
    'en': 'English'
}

# ============================================================
# 🎤 UPLOAD AND TRANSCRIBE
# ============================================================
print("\n📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Load audio
wav_np, sr = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True)
wav = torch.from_numpy(wav_np).float()

# VAD
speech_timestamps = get_speech_timestamps(wav, model_vad, sampling_rate=SAMPLING_RATE)
print(f"🎤 Found {len(speech_timestamps)} segment(s)\n")

for i, ts in enumerate(speech_timestamps):
    speech_chunk = wav_np[ts['start']:ts['end']]
    temp_path = f"segment_{i}.wav"
    sf.write(temp_path, speech_chunk, SAMPLING_RATE)

    # Try each language, pick best result
    best_result = None
    best_confidence = -999

    print(f"{'='*60}")
    print(f"📌 Segment {i+1} - Testing all languages...")
    print(f"{'='*60}")

    for lang_code, lang_name in SUPPORTED_LANGS.items():
        segments, info = model_asr.transcribe(
            temp_path,
            language=lang_code,  # 👈 FORCE language
            beam_size=5
        )
        segments_list = list(segments)

        if segments_list:
            text = " ".join([seg.text for seg in segments_list]).strip()
            avg_prob = sum(seg.avg_logprob for seg in segments_list) / len(segments_list)

            print(f"   {lang_name:10} → {text[:40]}... (conf: {avg_prob:.2f})")

            if avg_prob > best_confidence:
                best_confidence = avg_prob
                best_result = {
                    'language': lang_code,
                    'language_name': lang_name,
                    'text': text,
                    'confidence': avg_prob
                }

    # Print best result
    print(f"\n✅ BEST MATCH:")
    print(f"   🌐 Language: {best_result['language_name']} ({best_result['language']})")
    print(f"   📝 Text: {best_result['text']}")
    print(f"   📊 Confidence: {best_result['confidence']:.2f}\n")

In [ ]:
# ============================================================
# 🌐 SpeechBrain Language Detection (Fixed)
# ============================================================

# Fix torchaudio compatibility FIRST
!pip install -q torchaudio==2.1.0 speechbrain

import time
from google.colab import files
from speechbrain.inference.classifiers import EncoderClassifier

# Load language ID model
print("Loading SpeechBrain LangID model...")
lang_model = EncoderClassifier.from_hparams(
    source="speechbrain/lang-id-voxlingua107-ecapa",
    savedir="tmp_lang"
)
print("✓ LangID model loaded\n")

# ============================================================
# 🎤 UPLOAD AUDIO
# ============================================================
print("📁 Upload audio for language detection:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Detect language
start = time.time()
prediction = lang_model.classify_file(audio_path)
latency = (time.time() - start) * 1000

# Extract results
lang_code = prediction[3][0]
confidence = prediction[1].exp().item()

print(f"\n{'='*50}")
print(f"🌐 Language: {lang_code}")
print(f"📊 Confidence: {confidence:.1%}")
print(f"⏱️ Latency: {latency:.1f}ms")
print(f"{'='*50}")

lang_short = lang_code.split(":")[0].strip()
print(f"\n💡 For Whisper: language='{lang_short}'")

In [ ]:
# ============================================================
# 🌐 Fast Language Detection (Whisper - NO SpeechBrain)
# ============================================================
!pip install -q faster-whisper librosa

import time
import librosa
import torch
from google.colab import files
from faster_whisper import WhisperModel

SAMPLING_RATE = 16000

# Load Whisper (use "small" for fast detection, "large-v3" for accuracy)
print("Loading Whisper for language detection...")
model = WhisperModel("small", device="cuda", compute_type="float16")
print("✓ Model loaded\n")

# ============================================================
# 🎤 UPLOAD AUDIO
# ============================================================
print("📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Detect language (fast)
start = time.time()
_, info = model.transcribe(audio_path, beam_size=1, best_of=1)
detect_latency = (time.time() - start) * 1000

print(f"\n{'='*50}")
print(f"🌐 Language: {info.language}")
print(f"📊 Confidence: {info.language_probability:.1%}")
print(f"⏱️ Detection Latency: {detect_latency:.0f}ms")
print(f"{'='*50}")

# Now transcribe with detected language for accuracy
print(f"\n🔄 Transcribing with language='{info.language}'...")
start = time.time()
segments, _ = model.transcribe(audio_path, language=info.language, beam_size=5)
text = " ".join([seg.text for seg in segments]).strip()
transcribe_latency = (time.time() - start) * 1000

print(f"\n📝 Transcription: {text}")
print(f"⏱️ Transcription Latency: {transcribe_latency:.0f}ms")

In [ ]:
# ============================================================
# 🚀 Fast Language Detection (Whisper Tiny Model)
# ============================================================
!pip install -q faster-whisper librosa

import time
from google.colab import files
from faster_whisper import WhisperModel

# Load tiny model (~50-80ms detection)
print("Loading Whisper tiny model...")
model = WhisperModel("tiny", device="cuda", compute_type="float16")
print("✓ Model loaded\n")

# Upload audio
print("📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Detect language
start = time.time()
_, info = model.transcribe(audio_path, beam_size=1, best_of=1)
latency = (time.time() - start) * 1000

print(f"\n{'='*50}")
print(f"🌐 Language: {info.language}")
print(f"📊 Confidence: {info.language_probability:.1%}")
print(f"⏱️ Latency: {latency:.0f}ms")
print(f"{'='*50}")

In [ ]:
# ============================================================
# ⚡ fast-langdetect: Ultra-fast TEXT language detection
# ============================================================
!pip install -q fast-langdetect

from fast_langdetect import detect
import time

# Test with Indian language texts
test_texts = [
    "ನನಗೆ ಹೊಟ್ಟೆ ನೋವಾಗುತ್ತಿದೆ",        # Kannada
    "मुझे पेट में दर्द हो रहा है",            # Hindi
    "எனக்கு வயிற்று வலி",                   # Tamil
    "నాకు కడుపు నొప్పి",                     # Telugu
    "I have a stomach ache",                # English
]

print("⚡ fast-langdetect Test:\n")
print("="*60)

for text in test_texts:
    start = time.time()
    result = detect(text, model="auto", k=1)  # k=1 returns top 1 result
    latency = (time.time() - start) * 1000

    lang_code = result[0]['lang']
    confidence = result[0]['score']

    print(f"📝 Text: {text[:30]}...")
    print(f"   🌐 Language: {lang_code}")
    print(f"   📊 Confidence: {confidence:.1%}")
    print(f"   ⏱️ Latency: {latency:.2f}ms")
    print("-"*60)

In [ ]:
# ============================================================
# ⚡ 2-Stage: Quick Detect → Accurate Transcribe
# ============================================================
!pip install -q faster-whisper librosa soundfile

import time
import librosa
import soundfile as sf
from google.colab import files
from faster_whisper import WhisperModel

SUPPORTED_LANGUAGES = ['en', 'hi', 'te', 'kn', 'ta']
LANG_NAMES = {'en': 'English', 'hi': 'Hindi', 'te': 'Telugu', 'kn': 'Kannada', 'ta': 'Tamil'}
SAMPLING_RATE = 16000

# Load models
print("Loading models...")
model_tiny = WhisperModel("tiny", device="cuda", compute_type="float16")  # For detection
model_large = WhisperModel("large-v3", device="cuda", compute_type="float16")  # For transcription
print("✓ Models loaded\n")

# Upload
print("📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Stage 1: Quick detection with tiny model (first 2 seconds)
wav, sr = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True, duration=2.0)
sf.write("first_2s.wav", wav, SAMPLING_RATE)

print("\n🔍 Stage 1: Quick language detection...")
best_lang = None
best_conf = -999

for lang in SUPPORTED_LANGUAGES:
    segments, _ = model_tiny.transcribe("first_2s.wav", language=lang, beam_size=1)
    segs = list(segments)
    if segs:
        conf = sum(s.avg_logprob for s in segs) / len(segs)
        if conf > best_conf:
            best_conf = conf
            best_lang = lang

print(f"   → Detected: {LANG_NAMES[best_lang]} ({best_lang})")

# Stage 2: Full transcription with large model
print(f"\n📝 Stage 2: Transcribing with {LANG_NAMES[best_lang]}...")
start = time.time()
segments, _ = model_large.transcribe(audio_path, language=best_lang, beam_size=5)
text = " ".join([seg.text for seg in segments]).strip()
latency = (time.time() - start) * 1000

print(f"\n{'='*60}")
print(f"🌐 Language: {LANG_NAMES[best_lang]}")
print(f"📝 Text: {text}")
print(f"⏱️ Transcription Latency: {latency:.0f}ms")
print(f"{'='*60}")

In [ ]:
# ============================================================
# 🎯 Script-Based Language Detection (100% Accurate)
# ============================================================
!pip install -q faster-whisper librosa soundfile

import time
import librosa
import soundfile as sf
from google.colab import files
from faster_whisper import WhisperModel

SAMPLING_RATE = 16000

# Unicode script ranges - these NEVER overlap!
SCRIPT_RANGES = {
    'kn': [(0x0C80, 0x0CFF)],  # Kannada ಕನ್ನಡ
    'ta': [(0x0B80, 0x0BFF)],  # Tamil தமிழ்
    'te': [(0x0C00, 0x0C7F)],  # Telugu తెలుగు
    'hi': [(0x0900, 0x097F)],  # Hindi/Devanagari हिंदी
    'en': [(0x0041, 0x007A), (0x0030, 0x0039)],  # English A-z, 0-9
}
LANG_NAMES = {'en': 'English', 'hi': 'Hindi', 'te': 'Telugu', 'kn': 'Kannada', 'ta': 'Tamil'}

def detect_script(text):
    """Detect language from Unicode script - 100% accurate!"""
    script_counts = {lang: 0 for lang in SCRIPT_RANGES}

    for char in text:
        code = ord(char)
        for lang, ranges in SCRIPT_RANGES.items():
            for start, end in ranges:
                if start <= code <= end:
                    script_counts[lang] += 1
                    break

    # Return language with most characters
    best_lang = max(script_counts, key=script_counts.get)
    total = sum(script_counts.values())
    confidence = script_counts[best_lang] / max(total, 1)

    return best_lang, confidence, script_counts

# Load model
print("Loading Whisper large-v3...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Model loaded\n")

# Upload
print("📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Step 1: Transcribe with ALL indian languages (let Whisper pick)
print("\n📝 Transcribing...")
start = time.time()
segments, info = model.transcribe(audio_path, beam_size=5)
text = " ".join([seg.text for seg in segments]).strip()
stt_latency = (time.time() - start) * 1000

# Step 2: Detect actual language from SCRIPT (not audio)
lang, confidence, counts = detect_script(text)

print(f"\n{'='*60}")
print(f"📝 Transcription: {text}")
print(f"\n🔍 Detection:")
print(f"   • Whisper said: {info.language}")
print(f"   • Script says:  {LANG_NAMES[lang]} ({lang}) ← CORRECT")
print(f"   • Script counts: {counts}")
print(f"   • Confidence: {confidence:.0%}")
print(f"\n⏱️ Latency: {stt_latency:.0f}ms")
print(f"{'='*60}")

# If the script doesn't match Whisper's language, re-transcribe!
if lang != info.language and lang != 'en':
    print(f"\n🔄 Re-transcribing with correct language: {LANG_NAMES[lang]}...")
    start = time.time()
    segments, _ = model.transcribe(audio_path, language=lang, beam_size=5)
    text = " ".join([seg.text for seg in segments]).strip()
    latency = (time.time() - start) * 1000

    print(f"\n✅ CORRECTED:")
    print(f"📝 Text: {text}")
    print(f"⏱️ Latency: {latency:.0f}ms")

In [ ]:
# ============================================================
# 🎯 Audio-only Language Detection (100% Version Safe)
# ============================================================
!pip install -q faster-whisper librosa soundfile

import librosa
from faster_whisper import WhisperModel
from google.colab import files

SAMPLING_RATE = 16000

print("Loading Whisper model...")
model = WhisperModel("medium", device="cuda", compute_type="float16")
print("✓ Model loaded\n")

# Upload audio
print("📁 Upload audio file:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Load audio
audio, _ = librosa.load(audio_path, sr=SAMPLING_RATE)

# 🔍 Detect language (NO transcription)
result = model.detect_language(audio)

language = result[0]
confidence = result[1]   # <-- may be float OR dict in other versions

print("\n" + "="*50)
print(f"🌐 Detected Language : {language}")

# ✅ Handle both cases
if isinstance(confidence, dict):
    print(f"📊 Confidence       : {confidence[language]*100:.2f}%")
else:
    print(f"📊 Confidence       : {confidence*100:.2f}%")

print("="*50)


In [ ]:
# ============================================================
# 🎯 Audio-only Language Detection (Version Safe + Latency)
# ============================================================
!pip install -q faster-whisper librosa soundfile

import time
import librosa
from faster_whisper import WhisperModel
from google.colab import files

SAMPLING_RATE = 16000

print("Loading Whisper model...")
model = WhisperModel("medium", device="cuda", compute_type="float16")
print("✓ Model loaded\n")

# Upload audio
print("📁 Upload audio file:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Load audio
audio, _ = librosa.load(audio_path, sr=SAMPLING_RATE)

# 🔍 Detect language (NO transcription)
start_time = time.perf_counter()
result = model.detect_language(audio)
latency_ms = (time.perf_counter() - start_time) * 1000

language = result[0]
confidence = result[1]   # float OR dict (version dependent)

print("\n" + "="*50)
print(f"🌐 Detected Language : {language}")

# Handle both confidence formats
if isinstance(confidence, dict):
    conf = confidence[language] * 100
else:
    conf = confidence * 100

print(f"📊 Confidence       : {conf:.2f}%")
print(f"⏱️ Latency          : {latency_ms:.2f} ms")
print("="*50)


In [ ]:
# ============================================================
# 🎯 Language Detection: Whisper Primary + Unicode Fallback
# ============================================================
!pip install -q faster-whisper librosa soundfile

import time
import librosa
import soundfile as sf
from google.colab import files
from faster_whisper import WhisperModel

SAMPLING_RATE = 16000
CONFIDENCE_THRESHOLD = 0.85  # Use Whisper if >= this

# Unicode script ranges
SCRIPT_RANGES = {
    'kn': [(0x0C80, 0x0CFF)],  # Kannada ಕನ್ನಡ
    'ta': [(0x0B80, 0x0BFF)],  # Tamil தமிழ்
    'te': [(0x0C00, 0x0C7F)],  # Telugu తెలుగు
    'hi': [(0x0900, 0x097F)],  # Hindi हिंदी
    'en': [(0x0041, 0x007A)],  # English
}
LANG_NAMES = {'en': 'English', 'hi': 'Hindi', 'te': 'Telugu', 'kn': 'Kannada', 'ta': 'Tamil'}
SUPPORTED_LANGS = ['en', 'hi', 'te', 'kn', 'ta']

# Session state
session_language = None

def detect_from_script(text):
    """Fallback: Detect language from Unicode script."""
    script_counts = {lang: 0 for lang in SCRIPT_RANGES}

    for char in text:
        code = ord(char)
        for lang, ranges in SCRIPT_RANGES.items():
            for start, end in ranges:
                if start <= code <= end:
                    script_counts[lang] += 1
                    break

    best = max(script_counts, key=script_counts.get)
    total = sum(script_counts.values())
    confidence = script_counts[best] / max(total, 1)

    return best, confidence

def detect_language(audio_path, whisper_info, transcript, last_session_lang=None):
    """
    Smart language detection:
    1. Use Whisper if confidence >= 85%
    2. Else use Unicode script detection
    3. Else fallback to session language
    """
    whisper_lang = whisper_info.language
    whisper_conf = whisper_info.language_probability

    # Constrain to supported languages
    if whisper_lang not in SUPPORTED_LANGS:
        whisper_conf = 0  # Force fallback

    # Decision logic
    if whisper_conf >= CONFIDENCE_THRESHOLD:
        return {
            "language": whisper_lang,
            "method": "whisper",
            "confidence": whisper_conf
        }

    # Fallback to Unicode
    script_lang, script_conf = detect_from_script(transcript)

    if script_conf >= 0.5 and script_lang in SUPPORTED_LANGS:
        return {
            "language": script_lang,
            "method": "unicode",
            "confidence": script_conf
        }

    # Last resort: session language
    if last_session_lang:
        return {
            "language": last_session_lang,
            "method": "session",
            "confidence": 0.5
        }

    # Default to Hindi
    return {
        "language": "hi",
        "method": "default",
        "confidence": 0.0
    }

# ============================================================
# 🎤 LOAD MODEL
# ============================================================
print("Loading Whisper large-v3...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Model loaded\n")

# ============================================================
# 🎤 UPLOAD AND PROCESS
# ============================================================
print("📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# Step 1: Transcribe with Whisper (auto-detect)
print("\n🔄 Processing...")
start = time.time()
segments, info = model.transcribe(audio_path, beam_size=5)
transcript = " ".join([seg.text for seg in segments]).strip()
stt_latency = (time.time() - start) * 1000

# Step 2: Smart language detection
lang_start = time.time()
result = detect_language(audio_path, info, transcript, session_language)
lang_latency = (time.time() - lang_start) * 1000

# Update session
session_language = result["language"]

# Step 3: If method is not Whisper, re-transcribe with correct language
final_transcript = transcript
retranscribe_latency = 0

if result["method"] != "whisper":
    print(f"🔄 Re-transcribing with {LANG_NAMES[result['language']]}...")
    start = time.time()
    segments, _ = model.transcribe(audio_path, language=result["language"], beam_size=5)
    final_transcript = " ".join([seg.text for seg in segments]).strip()
    retranscribe_latency = (time.time() - start) * 1000

# ============================================================
# 📊 RESULTS
# ============================================================
print(f"\n{'='*60}")
print(f"📝 Transcription: {final_transcript}")
print(f"\n🌐 Language Detection:")
print(f"   • Whisper detected: {info.language} ({info.language_probability:.0%})")
print(f"   • Final decision:   {LANG_NAMES[result['language']]} ({result['language']})")
print(f"   • Method used:      {result['method'].upper()}")
print(f"   • Confidence:       {result['confidence']:.0%}")
print(f"\n⏱️ Latencies:")
print(f"   • STT:          {stt_latency:.0f}ms")
print(f"   • Lang detect:  {lang_latency:.2f}ms")
if retranscribe_latency > 0:
    print(f"   • Re-transcribe: {retranscribe_latency:.0f}ms")
print(f"   • Total:        {stt_latency + lang_latency + retranscribe_latency:.0f}ms")
print(f"{'='*60}")

In [ ]:
# ============================================================
# 🧠 LAYER-1 SIGNAL EXTRACTION (Pre-LLM Safety & Routing)
# ============================================================
import time
import re

# ============================================================
# 📚 KEYWORD DICTIONARIES (Multi-language)
# ============================================================

# Medical urgency keywords
URGENCY_HIGH = [
    # English
    'emergency', 'urgent', 'pain', 'severe', 'bleeding', 'chest', 'breathless',
    'accident', 'dying', 'help', 'attack', 'unconscious', 'critical', 'hurts',
    # Hindi
    'दर्द', 'तेज़', 'खून', 'सांस', 'दिल', 'बेहोश', 'मदद', 'गंभीर', 'जल्दी',
    # Kannada
    'ನೋವು', 'ರಕ್ತ', 'ಉಸಿರು', 'ಎದೆ', 'ತುರ್ತು', 'ಸಹಾಯ', 'ಬೇಗ', 'ಗಂಭೀರ',
    # Tamil
    'வலி', 'இரத்தம்', 'மூச்சு', 'நெஞ்சு', 'அவசரம்', 'உதவி',
    # Telugu
    'నొప్పి', 'రక్తం', 'ఊపిరి', 'ఛాతీ', 'అత్యవసర', 'సహాయం',
]

# Intent keywords
INTENT_MEDICAL = [
    'pain', 'fever', 'cough', 'headache', 'symptom', 'medicine', 'tablet', 'doctor',
    'treatment', 'diagnosis', 'sick', 'disease', 'health', 'body',
    'दर्द', 'बुखार', 'खांसी', 'सिरदर्द', 'दवाई', 'बीमार', 'इलाज',
    'ನೋವು', 'ಜ್ವರ', 'ಕೆಮ್ಮು', 'ತಲೆನೋವು', 'ಔಷಧಿ', 'ರೋಗ', 'ಚಿಕಿತ್ಸೆ',
    'வலி', 'காய்ச்சல்', 'இருமல்', 'தலைவலி', 'மருந்து',
]

INTENT_APPOINTMENT = [
    'appointment', 'book', 'schedule', 'doctor', 'visit', 'slot', 'time', 'meet',
    'अपॉइंटमेंट', 'बुक', 'डॉक्टर', 'मिलना', 'समय',
    'ಅಪಾಯಿಂಟ್ಮೆಂಟ್', 'ಬುಕ್', 'ಡಾಕ್ಟರ್', 'ಸಮಯ', 'ಭೇಟಿ',
]

INTENT_ADMIN = [
    'bill', 'payment', 'insurance', 'report', 'document', 'receipt', 'record',
    'बिल', 'पेमेंट', 'इंश्योरेंस', 'रिपोर्ट',
    'ಬಿಲ್', 'ಪೇಮೆಂಟ್', 'ರಿಪೋರ್ಟ್', 'ದಾಖಲೆ',
]

INTENT_GREETING = [
    'hello', 'hi', 'namaste', 'good morning', 'good evening', 'how are you',
    'नमस्ते', 'हेलो', 'कैसे हो',
    'ನಮಸ್ಕಾರ', 'ಹಲೋ', 'ಹೇಗಿದ್ದೀರಿ',
    'வணக்கம்',
    'నమస్కారం',
]

# Stress/urgency tone indicators
STRESS_INDICATORS = [
    'please', 'help', 'very', 'too much', 'can\'t', 'unable', 'worried', 'scared',
    'प्लीज', 'मदद', 'बहुत', 'नहीं हो रहा', 'चिंता', 'डर',
    'ದಯವಿಟ್ಟು', 'ಸಹಾಯ', 'ತುಂಬಾ', 'ಆಗುತ್ತಿಲ್ಲ', 'ಚಿಂತೆ',
]

# ============================================================
# 🧠 SIGNAL EXTRACTION FUNCTION
# ============================================================

def extract_layer1_signals(text, asr_confidence=0.9, language="en"):
    """
    Extract Layer-1 signals from ASR output.
    This happens BEFORE the LLM for routing & safety.

    Args:
        text: Transcribed text
        asr_confidence: Whisper language probability (0-1)
        language: Detected language code

    Returns:
        dict with intent, urgency, tone, confidence, escalate
    """
    start = time.time()
    text_lower = text.lower()

    # 1️⃣ INTENT DETECTION
    intent_scores = {
        'medical_query': sum(1 for kw in INTENT_MEDICAL if kw.lower() in text_lower),
        'appointment': sum(1 for kw in INTENT_APPOINTMENT if kw.lower() in text_lower),
        'admin': sum(1 for kw in INTENT_ADMIN if kw.lower() in text_lower),
        'greeting': sum(1 for kw in INTENT_GREETING if kw.lower() in text_lower),
    }
    intent = max(intent_scores, key=intent_scores.get) if max(intent_scores.values()) > 0 else "general"

    # 2️⃣ URGENCY DETECTION
    urgency_count = sum(1 for kw in URGENCY_HIGH if kw.lower() in text_lower)
    urgency = "high" if urgency_count >= 1 else "normal"

    # 3️⃣ TONE DETECTION
    stress_count = sum(1 for kw in STRESS_INDICATORS if kw.lower() in text_lower)
    tone = "stressed" if (stress_count >= 2 or urgency == "high") else "calm"

    # 4️⃣ CONFIDENCE ASSESSMENT
    confidence = "high" if asr_confidence >= 0.85 else "low"

    # 5️⃣ ESCALATION FLAG
    # Escalate if: high urgency OR low confidence on medical
    escalate = (
        urgency == "high" or
        (confidence == "low" and intent == "medical_query") or
        urgency_count >= 2
    )

    latency = (time.time() - start) * 1000

    return {
        "intent": intent,
        "urgency": urgency,
        "tone": tone,
        "confidence": confidence,
        "escalate": escalate,
        "language": language,
        "latency_ms": round(latency, 2),
        "_debug": {
            "intent_scores": intent_scores,
            "urgency_keywords": urgency_count,
            "stress_keywords": stress_count
        }
    }

# ============================================================
# 🎤 TEST WITH AUDIO UPLOAD
# ============================================================
from google.colab import files
from faster_whisper import WhisperModel

# Load model (skip if already loaded)
try:
    model
    print("✓ Model already loaded")
except:
    print("Loading Whisper...")
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")
    print("✓ Model loaded")

# Upload audio
print("\n📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# STT
print("\n🔄 Transcribing...")
stt_start = time.time()
segments, info = model.transcribe(audio_path, beam_size=5)
text = " ".join([seg.text for seg in segments]).strip()
stt_latency = (time.time() - stt_start) * 1000

# Layer-1 Extraction
signals = extract_layer1_signals(
    text=text,
    asr_confidence=info.language_probability,
    language=info.language
)

# ============================================================
# 📊 RESULTS
# ============================================================
print(f"\n{'='*60}")
print(f"📝 TRANSCRIPTION")
print(f"{'='*60}")
print(f"   Text: {text}")
print(f"   Language: {info.language} ({info.language_probability:.0%})")
print(f"   STT Latency: {stt_latency:.0f}ms")

print(f"\n{'='*60}")
print(f"🎯 LAYER-1 SIGNALS")
print(f"{'='*60}")
print(f"   ┌─────────────┬─────────────────────┐")
print(f"   │ Intent      │ {signals['intent']:19} │")
print(f"   │ Urgency     │ {signals['urgency']:19} │")
print(f"   │ Tone        │ {signals['tone']:19} │")
print(f"   │ Confidence  │ {signals['confidence']:19} │")
print(f"   │ Escalate    │ {'🚨 YES - ROUTE TO HUMAN' if signals['escalate'] else '✅ No':19} │")
print(f"   └─────────────┴─────────────────────┘")
print(f"   ⏱️ Layer-1 Latency: {signals['latency_ms']}ms")

print(f"\n{'='*60}")
print(f"🔍 DEBUG INFO")
print(f"{'='*60}")
print(f"   Intent scores: {signals['_debug']['intent_scores']}")
print(f"   Urgency keywords found: {signals['_debug']['urgency_keywords']}")
print(f"   Stress keywords found: {signals['_debug']['stress_keywords']}")

# Show what to pass to LLM
print(f"\n{'='*60}")
print(f"📤 PASS TO LLM")
print(f"{'='*60}")
llm_input = {
    "text": text,
    "language": signals["language"],
    "intent": signals["intent"],
    "urgency": signals["urgency"],
    "escalate": signals["escalate"]
}
print(f"   {llm_input}")

In [ ]:
# ============================================================
# 🧠 LAYER-1 + SAFETY GATE + POLICY ENGINE
# Complete Pre-LLM Pipeline
# ============================================================
import time
import re

# ============================================================
# 📚 KEYWORD DICTIONARIES (Multi-language)
# ============================================================

URGENCY_HIGH = [
    'emergency', 'urgent', 'pain', 'severe', 'bleeding', 'chest', 'breathless',
    'accident', 'dying', 'help', 'attack', 'unconscious', 'critical', 'hurts',
    'दर्द', 'तेज़', 'खून', 'सांस', 'दिल', 'बेहोश', 'मदद', 'गंभीर',
    'ನೋವು', 'ರಕ್ತ', 'ಉಸಿರು', 'ಎದೆ', 'ತುರ್ತು', 'ಸಹಾಯ', 'ಬೇಗ',
    'வலி', 'இரத்தம்', 'மூச்சு', 'நெஞ்சு', 'அவசரம்',
    'నొప్పి', 'రక్తం', 'ఊపిరి', 'ఛాతీ', 'అత్యవసర',
]

INTENT_MEDICAL = [
    'pain', 'fever', 'cough', 'headache', 'symptom', 'medicine', 'tablet', 'doctor',
    'treatment', 'diagnosis', 'sick', 'disease', 'body', 'stomach', 'throat',
    'दर्द', 'बुखार', 'खांसी', 'सिरदर्द', 'दवाई', 'बीमार',
    'ನೋವು', 'ಜ್ವರ', 'ಕೆಮ್ಮು', 'ತಲೆನೋವು', 'ಔಷಧಿ', 'ರೋಗ',
    'வலி', 'காய்ச்சல்', 'இருமல்', 'தலைவலி', 'மருந்து',
]

INTENT_APPOINTMENT = [
    'appointment', 'book', 'schedule', 'visit', 'slot', 'time', 'meet',
    'अपॉइंटमेंट', 'बुक', 'मिलना', 'समय',
    'ಅಪಾಯಿಂಟ್ಮೆಂಟ್', 'ಬುಕ್', 'ಸಮಯ', 'ಭೇಟಿ',
]

INTENT_ADMIN = [
    'bill', 'payment', 'insurance', 'report', 'document', 'receipt',
    'बिल', 'पेमेंट', 'रिपोर्ट',
    'ಬಿಲ್', 'ಪೇಮೆಂಟ್', 'ರಿಪೋರ್ಟ್',
]

INTENT_GREETING = [
    'hello', 'hi', 'namaste', 'good morning', 'how are you',
    'नमस्ते', 'हेलो',
    'ನಮಸ್ಕಾರ', 'ಹಲೋ',
    'வணக்கம்',
    'నమస్కారం',
]

STRESS_INDICATORS = [
    'please', 'help', 'very', 'too much', 'can\'t', 'worried', 'scared',
    'प्लीज', 'मदद', 'बहुत', 'चिंता',
    'ದಯವಿಟ್ಟು', 'ಸಹಾಯ', 'ತುಂಬಾ', 'ಚಿಂತೆ',
]

# Critical medical keywords that ALWAYS escalate
CRITICAL_SYMPTOMS = [
    'chest pain', 'heart attack', 'can\'t breathe', 'unconscious', 'suicide',
    'bleeding heavily', 'stroke', 'seizure', 'overdose',
    'छाती में दर्द', 'दिल का दौरा', 'सांस नहीं',
    'ಎದೆ ನೋವು', 'ಹೃದಯಾಘಾತ', 'ಉಸಿರಾಡಲು',
]

# Session state for tracking repeated queries
query_history = []

# ============================================================
# 🧠 STEP 2: LAYER-1 SIGNAL EXTRACTION
# ============================================================

def extract_layer1_signals(text, asr_confidence=0.9, language="en"):
    """Extract intent, urgency, tone, confidence from ASR output."""
    start = time.time()
    text_lower = text.lower()

    # Intent detection
    intent_scores = {
        'medical_query': sum(1 for kw in INTENT_MEDICAL if kw.lower() in text_lower),
        'appointment': sum(1 for kw in INTENT_APPOINTMENT if kw.lower() in text_lower),
        'admin': sum(1 for kw in INTENT_ADMIN if kw.lower() in text_lower),
        'greeting': sum(1 for kw in INTENT_GREETING if kw.lower() in text_lower),
    }
    intent = max(intent_scores, key=intent_scores.get) if max(intent_scores.values()) > 0 else "general"

    # Urgency & tone
    urgency_count = sum(1 for kw in URGENCY_HIGH if kw.lower() in text_lower)
    urgency = "high" if urgency_count >= 1 else "normal"

    stress_count = sum(1 for kw in STRESS_INDICATORS if kw.lower() in text_lower)
    tone = "stressed" if (stress_count >= 2 or urgency == "high") else "calm"

    confidence = "high" if asr_confidence >= 0.85 else "low"

    latency = (time.time() - start) * 1000

    return {
        "intent": intent,
        "urgency": urgency,
        "tone": tone,
        "confidence": confidence,
        "language": language,
        "latency_ms": round(latency, 2),
        "_raw": {"urgency_count": urgency_count, "stress_count": stress_count}
    }

# ============================================================
# 🚨 STEP 3: SAFETY & ESCALATION GATE (Rule-based)
# ============================================================

def safety_gate(text, signals):
    """
    Hard safety rules BEFORE LLM.
    Returns escalation decision and reason.
    """
    start = time.time()
    text_lower = text.lower()

    escalate = False
    reasons = []

    # Rule 1: Critical symptoms → ALWAYS escalate
    for symptom in CRITICAL_SYMPTOMS:
        if symptom.lower() in text_lower:
            escalate = True
            reasons.append(f"Critical symptom: {symptom}")
            break

    # Rule 2: Chest pain OR breathlessness → escalate
    if any(kw in text_lower for kw in ['chest', 'breathless', 'ಎದೆ', 'ಉಸಿರು', 'छाती', 'सांस']):
        if signals["intent"] == "medical_query":
            escalate = True
            reasons.append("Chest/breathing issue detected")

    # Rule 3: Low confidence + medical query → escalate
    if signals["confidence"] == "low" and signals["intent"] == "medical_query":
        escalate = True
        reasons.append("Low ASR confidence on medical query")

    # Rule 4: High urgency → escalate
    if signals["urgency"] == "high" and signals["_raw"]["urgency_count"] >= 2:
        escalate = True
        reasons.append("Multiple urgency indicators")

    # Rule 5: Repeated similar query → escalate (user struggling)
    global query_history
    if len(query_history) >= 2:
        if text_lower in [q.lower() for q in query_history[-2:]]:
            escalate = True
            reasons.append("Repeated query - user may be struggling")
    query_history.append(text)
    if len(query_history) > 5:
        query_history.pop(0)

    latency = (time.time() - start) * 1000

    return {
        "escalate": escalate,
        "reasons": reasons,
        "action": "ROUTE_TO_HUMAN" if escalate else "PROCEED_TO_LLM",
        "latency_ms": round(latency, 2)
    }

# ============================================================
# ⚙️ STEP 4: POLICY ENGINE (Hospital Admin Controls)
# ============================================================

# Default hospital policy (can be configured via admin panel)
HOSPITAL_POLICY = {
    "name": "Apollo Hospital",
    "tone": "formal",           # formal / friendly
    "response_length": "short", # short / detailed
    "always_recommend_doctor": True,
    "language_fallback": "en",
    "disclaimer_required": True,
    "max_response_words": 50,
    "emergency_number": "108",
}

def apply_policy(signals, safety_result):
    """
    Apply hospital-specific policies.
    Returns system constraints for LLM.
    """
    start = time.time()
    policy = HOSPITAL_POLICY

    # Build system prompt constraints
    constraints = []

    # Tone
    if policy["tone"] == "formal":
        constraints.append("Use formal, professional language")
    else:
        constraints.append("Use warm, friendly language")

    # Response length
    if policy["response_length"] == "short":
        constraints.append(f"Keep response under {policy['max_response_words']} words")
    else:
        constraints.append("Provide detailed explanations")

    # Doctor recommendation
    if policy["always_recommend_doctor"]:
        constraints.append("Always suggest consulting a doctor for medical queries")

    # Disclaimer
    if policy["disclaimer_required"] and signals["intent"] == "medical_query":
        constraints.append("Include medical disclaimer")

    # Language
    response_language = signals["language"]
    if signals["confidence"] == "low":
        response_language = policy["language_fallback"]

    # Emergency info
    if safety_result["escalate"]:
        constraints.append(f"Mention emergency number: {policy['emergency_number']}")

    latency = (time.time() - start) * 1000

    return {
        "hospital": policy["name"],
        "response_language": response_language,
        "constraints": constraints,
        "system_prompt_additions": " | ".join(constraints),
        "latency_ms": round(latency, 2)
    }

# ============================================================
# 🎤 TEST: COMPLETE PIPELINE
# ============================================================
from google.colab import files
from faster_whisper import WhisperModel

# Load model
try:
    model
    print("✓ Model already loaded")
except:
    print("Loading Whisper...")
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")
    print("✓ Model loaded")

# Upload
print("\n📁 Upload audio:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# ===== PIPELINE =====
total_start = time.time()

# Step 1: STT
print("\n🔄 Step 1: Transcribing...")
stt_start = time.time()
segments, info = model.transcribe(audio_path, beam_size=5)
text = " ".join([seg.text for seg in segments]).strip()
stt_latency = (time.time() - stt_start) * 1000

# Step 2: Layer-1 Signals
print("🧠 Step 2: Extracting signals...")
signals = extract_layer1_signals(text, info.language_probability, info.language)

# Step 3: Safety Gate
print("🚨 Step 3: Safety check...")
safety = safety_gate(text, signals)

# Step 4: Policy Engine
print("⚙️ Step 4: Applying policy...")
policy = apply_policy(signals, safety)

total_latency = (time.time() - total_start) * 1000

# ============================================================
# 📊 RESULTS
# ============================================================
print(f"\n{'='*70}")
print(f"📝 STEP 1: TRANSCRIPTION")
print(f"{'='*70}")
print(f"   Text: {text}")
print(f"   Language: {info.language} ({info.language_probability:.0%})")
print(f"   ⏱️ {stt_latency:.0f}ms")

print(f"\n{'='*70}")
print(f"🧠 STEP 2: LAYER-1 SIGNALS")
print(f"{'='*70}")
print(f"   Intent:     {signals['intent']}")
print(f"   Urgency:    {signals['urgency']}")
print(f"   Tone:       {signals['tone']}")
print(f"   Confidence: {signals['confidence']}")
print(f"   ⏱️ {signals['latency_ms']}ms")

print(f"\n{'='*70}")
print(f"🚨 STEP 3: SAFETY GATE")
print(f"{'='*70}")
print(f"   Action:  {safety['action']}")
print(f"   Escalate: {'🚨 YES' if safety['escalate'] else '✅ No'}")
if safety['reasons']:
    print(f"   Reasons:")
    for r in safety['reasons']:
        print(f"      • {r}")
print(f"   ⏱️ {safety['latency_ms']}ms")

print(f"\n{'='*70}")
print(f"⚙️ STEP 4: POLICY ENGINE")
print(f"{'='*70}")
print(f"   Hospital: {policy['hospital']}")
print(f"   Language: {policy['response_language']}")
print(f"   Constraints:")
for c in policy['constraints']:
    print(f"      • {c}")
print(f"   ⏱️ {policy['latency_ms']}ms")

print(f"\n{'='*70}")
print(f"📤 FINAL OUTPUT → Send to LLM")
print(f"{'='*70}")
llm_input = {
    "text": text,
    "language": policy['response_language'],
    "intent": signals['intent'],
    "urgency": signals['urgency'],
    "escalate": safety['escalate'],
    "system_constraints": policy['system_prompt_additions']
}
for k, v in llm_input.items():
    print(f"   {k}: {v}")

print(f"\n{'='*70}")
print(f"⏱️ TOTAL PRE-LLM LATENCY: {total_latency:.0f}ms")
print(f"   (STT: {stt_latency:.0f}ms | Signals: {signals['latency_ms']}ms | Safety: {safety['latency_ms']}ms | Policy: {policy['latency_ms']}ms)")
print(f"{'='*70}")

In [ ]:
# ============================================================
# 🎙️ COMPLETE PIPELINE: VAD → STT → Signals → Safety → LLM
# Using Sarvam-1 LOCAL (No API)
# ============================================================

# ===== CELL 1: INSTALL DEPENDENCIES =====
!pip install -q faster-whisper transformers accelerate torch librosa soundfile

# ===== CELL 2: IMPORTS =====
import time
import torch
import librosa
import soundfile as sf
from google.colab import files
from faster_whisper import WhisperModel
from transformers import AutoModelForCausalLM, AutoTokenizer

SAMPLING_RATE = 16000

# ============================================================
# 🔧 LOAD MODELS
# ============================================================
print("Loading Whisper STT...")
stt_model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("✓ Whisper loaded")

print("\nLoading Silero VAD...")
torch.set_num_threads(1)
vad_model, utils = torch.hub.load('snakers4/silero-vad', model='silero_vad', force_reload=False, onnx=False)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils
print("✓ VAD loaded")

print("\nLoading Sarvam-1 LLM (this takes 2-3 mins)...")
llm_tokenizer = AutoTokenizer.from_pretrained("sarvamai/sarvam-1")
llm_model = AutoModelForCausalLM.from_pretrained(
    "sarvamai/sarvam-1",
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✓ Sarvam-1 loaded")

print("\n" + "="*60)
print("✅ ALL MODELS READY!")
print("="*60)

# ============================================================
# 📚 KEYWORD DICTIONARIES
# ============================================================
URGENCY_HIGH = [
    'emergency', 'urgent', 'pain', 'severe', 'bleeding', 'chest', 'breathless',
    'accident', 'dying', 'help', 'attack', 'unconscious', 'critical',
    'दर्द', 'खून', 'सांस', 'दिल', 'बेहोश', 'मदद',
    'ನೋವು', 'ರಕ್ತ', 'ಉಸಿರು', 'ಎದೆ', 'ತುರ್ತು', 'ಸಹಾಯ',
    'வலி', 'இரத்தம்', 'மூச்சு', 'நெஞ்சு', 'அவசரம்',
    'నొప్పి', 'రక్తం', 'ఊపిరి', 'ఛాతీ', 'అత్యవసర',
]

INTENT_MEDICAL = [
    'pain', 'fever', 'cough', 'headache', 'symptom', 'medicine', 'doctor',
    'दर्द', 'बुखार', 'खांसी', 'दवाई', 'बीमार',
    'ನೋವು', 'ಜ್ವರ', 'ಕೆಮ್ಮು', 'ಔಷಧಿ', 'ರೋಗ',
    'வலி', 'காய்ச்சல்', 'மருந்து',
]

INTENT_APPOINTMENT = ['appointment', 'book', 'schedule', 'visit', 'अपॉइंटमेंट', 'ಅಪಾಯಿಂಟ್ಮೆಂಟ್']
INTENT_ADMIN = ['bill', 'payment', 'insurance', 'report', 'बिल', 'ಬಿಲ್']
INTENT_GREETING = ['hello', 'hi', 'namaste', 'नमस्ते', 'ನಮಸ್ಕಾರ', 'வணக்கம்']
STRESS_INDICATORS = ['please', 'help', 'very', 'worried', 'मदद', 'ಸಹಾಯ', 'ತುಂಬಾ']
CRITICAL_SYMPTOMS = ['chest pain', 'heart attack', 'can\'t breathe', 'unconscious', 'ಎದೆ ನೋವು', 'छाती में दर्द']

LANG_NAMES = {'en': 'English', 'hi': 'Hindi', 'kn': 'Kannada', 'ta': 'Tamil', 'te': 'Telugu'}

# Hospital policy
HOSPITAL_POLICY = {
    "name": "Apollo Hospital",
    "tone": "formal",
    "response_length": "short",
    "always_recommend_doctor": True,
    "max_response_words": 50,
    "emergency_number": "108",
}

# Session tracking
query_history = []

# ============================================================
# 🧠 STEP 2: LAYER-1 SIGNAL EXTRACTION
# ============================================================
def extract_layer1_signals(text, asr_confidence=0.9, language="en"):
    start = time.time()
    text_lower = text.lower()

    intent_scores = {
        'medical_query': sum(1 for kw in INTENT_MEDICAL if kw.lower() in text_lower),
        'appointment': sum(1 for kw in INTENT_APPOINTMENT if kw.lower() in text_lower),
        'admin': sum(1 for kw in INTENT_ADMIN if kw.lower() in text_lower),
        'greeting': sum(1 for kw in INTENT_GREETING if kw.lower() in text_lower),
    }
    intent = max(intent_scores, key=intent_scores.get) if max(intent_scores.values()) > 0 else "general"

    urgency_count = sum(1 for kw in URGENCY_HIGH if kw.lower() in text_lower)
    urgency = "high" if urgency_count >= 1 else "normal"

    stress_count = sum(1 for kw in STRESS_INDICATORS if kw.lower() in text_lower)
    tone = "stressed" if (stress_count >= 2 or urgency == "high") else "calm"

    confidence = "high" if asr_confidence >= 0.85 else "low"

    return {
        "intent": intent, "urgency": urgency, "tone": tone,
        "confidence": confidence, "language": language,
        "latency_ms": round((time.time() - start) * 1000, 2),
        "_raw": {"urgency_count": urgency_count, "stress_count": stress_count}
    }

# ============================================================
# 🚨 STEP 3: SAFETY GATE
# ============================================================
def safety_gate(text, signals):
    global query_history
    start = time.time()
    text_lower = text.lower()

    escalate = False
    reasons = []

    for symptom in CRITICAL_SYMPTOMS:
        if symptom.lower() in text_lower:
            escalate = True
            reasons.append(f"Critical: {symptom}")

    if any(kw in text_lower for kw in ['chest', 'breathless', 'ಎದೆ', 'ಉಸಿರು', 'छाती', 'सांस']):
        if signals["intent"] == "medical_query":
            escalate = True
            reasons.append("Chest/breathing issue")

    if signals["confidence"] == "low" and signals["intent"] == "medical_query":
        escalate = True
        reasons.append("Low confidence on medical")

    if signals["urgency"] == "high" and signals["_raw"]["urgency_count"] >= 2:
        escalate = True
        reasons.append("Multiple urgency words")

    query_history.append(text)
    if len(query_history) > 5: query_history.pop(0)

    return {
        "escalate": escalate, "reasons": reasons,
        "action": "ROUTE_TO_HUMAN" if escalate else "PROCEED_TO_LLM",
        "latency_ms": round((time.time() - start) * 1000, 2)
    }

# ============================================================
# ⚙️ STEP 4: POLICY ENGINE
# ============================================================
def apply_policy(signals, safety):
    start = time.time()
    policy = HOSPITAL_POLICY

    constraints = []
    constraints.append("Use formal, professional language" if policy["tone"] == "formal" else "Use friendly language")
    constraints.append(f"Keep response under {policy['max_response_words']} words")
    if policy["always_recommend_doctor"]: constraints.append("Recommend consulting doctor")
    if signals["intent"] == "medical_query": constraints.append("Include disclaimer")
    if safety["escalate"]: constraints.append(f"Mention emergency: {policy['emergency_number']}")

    return {
        "hospital": policy["name"],
        "response_language": signals["language"],
        "constraints": constraints,
        "latency_ms": round((time.time() - start) * 1000, 2)
    }

# ============================================================
# 📝 STEP 5: PROMPT CONSTRUCTION
# ============================================================
def construct_prompt(text, signals, safety, policy):
    start = time.time()
    lang_name = LANG_NAMES.get(policy['response_language'], 'Hindi')

    system_prompt = f"""You are a hospital voice assistant for {policy['hospital']}.

RULES:
- Respond ONLY in {lang_name}
- Keep response under 50 words (must be short for voice)
- Be formal and professional
- Never give specific medication dosages
- Always recommend consulting a doctor for medical issues
{"- URGENT: This is an emergency. Mention to call 108 immediately." if safety['escalate'] else ""}

User is asking about: {signals['intent']}
Urgency level: {signals['urgency']}"""

    return {
        "system_prompt": system_prompt,
        "user_message": text,
        "latency_ms": round((time.time() - start) * 1000, 2)
    }

# ============================================================
# 🚀 ALTERNATIVE: GEMMA-2B (Faster, Free, Good for Indian langs)
# ============================================================
!pip install -q transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading Gemma-2B...")
llm_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
llm_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✓ Gemma-2B loaded")

def call_gemma(system_prompt, user_message):
    start = time.time()

    prompt = f"""<start_of_turn>user
{system_prompt}

{user_message}<end_of_turn>
<start_of_turn>model
"""

    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True,
        )

    new_tokens = outputs[0][input_length:]
    response = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "response": response,
        "latency_ms": round((time.time() - start) * 1000, 2)
    }

# Test
test = call_gemma(
    "You are a hospital assistant. Respond in Kannada. Keep response under 30 words.",
    "ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು"
)
print(f"Response: {test['response']}")
print(f"Latency: {test['latency_ms']:.0f}ms")
# ============================================================
# 🧪 QUICK TEST
# ============================================================
test_response = call_sarvam_local(
    "You are a hospital assistant. Respond in Kannada. Keep it short.",
    "ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು"
)
print(f"Test response: {test_response['response']}")
print(f"Latency: {test_response['latency_ms']:.0f}ms")

# ============================================================
# 🎤 TEST: COMPLETE PIPELINE
# ============================================================
print("\n📁 Upload audio for full pipeline test:")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

total_start = time.time()

# Step 1: STT
print("\n🔄 Step 1: Transcribing...")
stt_start = time.time()
segments, info = stt_model.transcribe(audio_path, beam_size=5)
text = " ".join([seg.text for seg in segments]).strip()
stt_latency = (time.time() - stt_start) * 1000

# Step 2: Layer-1 Signals
print("🧠 Step 2: Extracting signals...")
signals = extract_layer1_signals(text, info.language_probability, info.language)

# Step 3: Safety Gate
print("🚨 Step 3: Safety check...")
safety = safety_gate(text, signals)

# Step 4: Policy Engine
print("⚙️ Step 4: Applying policy...")
policy = apply_policy(signals, safety)

# Step 5: Prompt Construction
print("📝 Step 5: Building prompt...")
prompt = construct_prompt(text, signals, safety, policy)

# Step 6: LLM Response
print("🤖 Step 6: Generating response...")
llm_result = call_sarvam_local(prompt["system_prompt"], prompt["user_message"])

total_latency = (time.time() - total_start) * 1000

# ============================================================
# 📊 RESULTS
# ============================================================
print(f"\n{'='*70}")
print(f"📝 USER INPUT (Step 1)")
print(f"{'='*70}")
print(f"   Text: {text}")
print(f"   Language: {info.language} ({info.language_probability:.0%})")
print(f"   ⏱️ STT: {stt_latency:.0f}ms")

print(f"\n{'='*70}")
print(f"🎯 SIGNALS (Step 2-4)")
print(f"{'='*70}")
print(f"   Intent: {signals['intent']} | Urgency: {signals['urgency']} | Tone: {signals['tone']}")
print(f"   Escalate: {'🚨 YES' if safety['escalate'] else '✅ No'}")
if safety['reasons']:
    print(f"   Reasons: {', '.join(safety['reasons'])}")

print(f"\n{'='*70}")
print(f"🤖 LLM RESPONSE (Step 6)")
print(f"{'='*70}")
print(f"   {llm_result['response']}")
print(f"   ⏱️ LLM: {llm_result['latency_ms']:.0f}ms")

print(f"\n{'='*70}")
print(f"⏱️ TOTAL LATENCY: {total_latency:.0f}ms")
print(f"{'='*70}")
print(f"   STT:      {stt_latency:.0f}ms")
print(f"   Signals:  {signals['latency_ms']}ms")
print(f"   Safety:   {safety['latency_ms']}ms")
print(f"   Policy:   {policy['latency_ms']}ms")
print(f"   Prompt:   {prompt['latency_ms']}ms")
print(f"   LLM:      {llm_result['latency_ms']:.0f}ms")
print(f"{'='*70}")

In [ ]:
import os
# ============================================================
# 🦙 LLAMA-3.1-8B-INSTRUCT (INT4)
# ============================================================
!pip install -q transformers accelerate bitsandbytes huggingface_hub

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Login
login(token=os.environ["HF_TOKEN"])

torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading LLaMA-3.1-8B...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
print("✓ LLaMA Loaded!")

# ============================================================
# 🧪 KANNADA TEST
# ============================================================
def generate(prompt, lang="kn"):
    if lang == "kn":
        system = "ನೀವು ಅಪೊಲೊ ಆಸ್ಪತ್ರೆಯ ಸಹಾಯಕ. ಕನ್ನಡದಲ್ಲಿ ಮಾತ್ರ ಉತ್ತರಿಸಿ. ಡಾ. ರಾಜ್ ಸೋಮ-ಶುಕ್ರ 10-1, 4-7 ಲಭ್ಯ. ತುರ್ತು: 108."
    else:
        system = "You are Apollo Hospital assistant. Dr. Raj available Mon-Fri 10-1, 4-7. Emergency: 108."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)

    outputs = model.generate(inputs, max_new_tokens=80, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# Test
print("\n🧪 KANNADA TEST:")
print(f"📝 Input: ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು")
print(f"🤖 Output: {generate('ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು')}")

In [ ]:
# ============================================================
# CELL 1: INSTALL (Run this, then restart runtime)
# ============================================================
!pip install -U bitsandbytes accelerate transformers huggingface_hub

In [ ]:
import os
# ============================================================
# CELL 2: LLAMA-3.1-8B KANNADA
# ============================================================
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

login(token=os.environ["HF_TOKEN"])
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print("Loading LLaMA-3.1-8B...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
print("✓ Loaded!")

def generate(prompt):
    system = """You are Apollo Hospital's voice assistant.
RESPOND ONLY IN KANNADA SCRIPT (ಕನ್ನಡ ಲಿಪಿ). NO ENGLISH.
ಡಾ. ರಾಜ್: ಸೋಮ-ಶುಕ್ರ 10-1, 4-7. ತುರ್ತು: 108."""

    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=80, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print(generate("ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು"))

In [ ]:
# ============================================================
# 🚀 MISTRAL-7B - KANNADA (One Cell, Works on T4!)
# ============================================================
!pip install -q transformers accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print("Loading Mistral-7B...")
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
)
print("✓ Loaded!")

def generate(prompt):
    system = """You are Apollo Hospital's voice assistant.

RULES:
1. Respond ONLY in Kannada script (Unicode characters like ನಮಸ್ಕಾರ)
2. NO English in your response
3. Keep response under 50 words

Hospital Info:
- Dr. Raj: Monday-Friday, 10AM-1PM, 4PM-7PM
- Emergency: 108"""

    messages = [{"role": "user", "content": f"{system}\n\nUser query: {prompt}"}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    outputs = model.generate(inputs, max_new_tokens=80, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# Test
print("\n🧪 TEST:")
for q in ["ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು", "ನನಗೆ ಹೊಟ್ಟೆ ನೋವು"]:
    print(f"\n📝 {q}")
    print(f"🤖 {generate(q)}")

In [ ]:
import os
# ============================================================
# 🦙 LLAMA-3.1-8B - FULL FREEDOM (Kannada Response)
# ============================================================
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

login(token=os.environ["HF_TOKEN"])
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print("Loading LLaMA-3.1-8B...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
print("✓ Loaded!")

def generate(prompt):
    # Simple system prompt - only language constraint
    system = "You are a helpful assistant. Always respond in Kannada script only."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(
        inputs,
        max_new_tokens=150,  # More tokens for longer responses
        temperature=0.8,     # More creative
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# Test
print("\n🧪 TEST:")
for q in ["ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು", "ನನಗೆ ಹೊಟ್ಟೆ ನೋವು", "ಹವಾಮಾನ ಹೇಗಿದೆ?"]:
    print(f"\n📝 {q}")
    print(f"🤖 {generate(q)}")


In [ ]:
!pip install -U bitsandbytes accelerate transformers huggingface_hub

In [ ]:
import os
# ============================================================
# 🦙 LLAMA-3.1-8B - FULL FREEDOM
# ============================================================
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

login(token=os.environ["HF_TOKEN"])
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print("Loading LLaMA-3.1-8B...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
print("✓ Loaded!")

def generate(prompt):
    system = "You are a helpful assistant. Always respond in Kannada script only."
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=150, temperature=0.8, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# Test
for q in ["ನನಗೆ ಡಾಕ್ಟರ್ ಅಪಾಯಿಂಟ್ಮೆಂಟ್ ಬೇಕು", "ಭಾರತ ಎಲ್ಲಿದೆ?"]:
    print(f"\n📝 {q}")
    print(f"🤖 {generate(q)}")

In [ ]:
import os
# ============================================================
# 🔊 INDIC PARLER-TTS (With HF Login)
# ============================================================
!pip install -q git+https://github.com/huggingface/parler-tts.git soundfile

import torch
import soundfile as sf
from IPython.display import Audio, display
from huggingface_hub import login

# Login first!
login(token=os.environ["HF_TOKEN"])

from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Loading Indic Parler-TTS...")
model = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
print("✓ Loaded!")

def tts(text, style="calm"):
    desc = "A female speaker delivers a calm and professional speech." if style == "calm" else "A female speaker delivers urgent speech with faster pace."
    desc_ids = description_tokenizer(desc, return_tensors="pt").to(device)
    prompt_ids = tokenizer(text, return_tensors="pt").to(device)
    audio = model.generate(input_ids=desc_ids.input_ids, attention_mask=desc_ids.attention_mask, prompt_input_ids=prompt_ids.input_ids, prompt_attention_mask=prompt_ids.attention_mask)
    return audio.cpu().numpy().squeeze(), model.config.sampling_rate

# Test
audio, sr = tts("ನಮಸ್ಕಾರ, ಡಾಕ್ಟರ್ ರಾಜ್ ಲಭ್ಯರಿದ್ದಾರೆ.")
display(Audio(audio, rate=sr))

In [ ]:
# ============================================================
# 🔊 EXPLICIT KANNADA TTS
# ============================================================

def tts_kannada(text):
    # Description explicitly mentioning Kannada
    desc = "A Kannada female speaker delivers a very slow and clear speech with moderate pitch. The speaker speaks in Kannada language with clear pronunciation. The recording is of very high quality."

    desc_ids = description_tokenizer(desc, return_tensors="pt").to(device)
    prompt_ids = tokenizer(text, return_tensors="pt").to(device)

    audio = model.generate(
        input_ids=desc_ids.input_ids,
        attention_mask=desc_ids.attention_mask,
        prompt_input_ids=prompt_ids.input_ids,
        prompt_attention_mask=prompt_ids.attention_mask
    )

    return audio.cpu().numpy().squeeze(), model.config.sampling_rate

# Test - make sure text is in Kannada script
kannada_text = "ಈ ಲಕ್ಷಣ ಗಂಭೀರವಾಗಿರಬಹುದು. ದಯವಿಟ್ಟು ತಕ್ಷಣ ವೈದ್ಯರನ್ನು ಸಂಪರ್ಕಿಸಿ."
print(f"📝 Text: {kannada_text}")

audio, sr = tts_kannada(kannada_text)
display(Audio(audio, rate=sr))

In [ ]:
# ============================================================
# ⚡ OPTIMIZED TTS (Lower Latency)
# ============================================================
import torch
import time

# 1️⃣ Pre-cache description tokenization (saves ~20ms per call)
CACHED_DESCRIPTIONS = {}

def cache_description(style, desc):
    """Pre-tokenize descriptions once"""
    if style not in CACHED_DESCRIPTIONS:
        CACHED_DESCRIPTIONS[style] = description_tokenizer(desc, return_tensors="pt").to(device)
    return CACHED_DESCRIPTIONS[style]

# Pre-cache common descriptions
cache_description("kannada_calm", "A Kannada female speaker delivers slow clear speech with moderate pitch.")
cache_description("kannada_fast", "A Kannada female speaker delivers moderately paced clear speech.")

# 2️⃣ Optimized TTS function
@torch.inference_mode()  # Faster than torch.no_grad()
def tts_fast(text, style="kannada_calm"):
    start = time.time()

    # Use cached description
    desc_ids = CACHED_DESCRIPTIONS[style]
    prompt_ids = tokenizer(text, return_tensors="pt").to(device)

    # Generate with optimized settings
    audio = model.generate(
        input_ids=desc_ids.input_ids,
        attention_mask=desc_ids.attention_mask,
        prompt_input_ids=prompt_ids.input_ids,
        prompt_attention_mask=prompt_ids.attention_mask,
        min_new_tokens=10,  # Minimum audio length
    )

    latency = (time.time() - start) * 1000
    print(f"⏱️ TTS Latency: {latency:.0f}ms")

    return audio.cpu().numpy().squeeze(), model.config.sampling_rate

# 3️⃣ Warm up the model (first call is always slow)
print("🔥 Warming up...")
_ = tts_fast("ಟೆಸ್ಟ್")

# Test
print("\n🧪 SPEED TEST:")
audio, sr = tts_fast("ಈ ಲಕ್ಷಣ ಗಂಭೀರವಾಗಿರಬಹುದು. ದಯವಿಟ್ಟು ತಕ್ಷಣ ವೈದ್ಯರನ್ನು ಸಂಪರ್ಕಿಸಿ.")
display(Audio(audio, rate=sr))

In [ ]:
# ============================================================
# ⚡ EDGE TTS (COLAB FIX)
# ============================================================
!pip install -q edge-tts nest_asyncio

import edge_tts
import asyncio
import nest_asyncio
import time
from IPython.display import Audio, display

# Fix for Colab
nest_asyncio.apply()

# Voices
KANNADA_VOICE = "kn-IN-SapnaNeural"
HINDI_VOICE = "hi-IN-SwaraNeural"
TAMIL_VOICE = "ta-IN-PallaviNeural"
TELUGU_VOICE = "te-IN-ShrutiNeural"

def speak(text, voice=KANNADA_VOICE, rate="-10%"):
    """Fast TTS using Edge"""
    start = time.time()

    async def generate():
        communicate = edge_tts.Communicate(text, voice, rate=rate)
        await communicate.save("output.mp3")

    asyncio.get_event_loop().run_until_complete(generate())

    latency = (time.time() - start) * 1000
    print(f"⏱️ TTS Latency: {latency:.0f}ms")
    display(Audio("output.mp3"))

# ============================================================
# 🧪 TEST
# ============================================================
print("🧪 EDGE TTS TEST:\n")

print("📝 Kannada:")
speak("ನಮಸ್ಕಾರ. ಡಾಕ್ಟರ್ ರಾಜ್ ಲಭ್ಯರಿದ್ದಾರೆ.")

print("\n📝 Hindi:")
speak("नमस्ते। डॉक्टर राज उपलब्ध हैं।", voice=HINDI_VOICE)

print("\n📝 Slower:")
speak("ದಯವಿಟ್ಟು ವೈದ್ಯರನ್ನು ಸಂಪರ್ಕಿಸಿ.", rate="-25%")